In [1]:
# https://qwen.readthedocs.io/en/latest/framework/LlamaIndex.html

from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def completion_to_prompt(completion):
    return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"


def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n" + prompt

    prompt = prompt + "<|im_start|>assistant\n"

    return prompt


def setup(model, tokenizer, embed_model, input_dir = None, input_files = None):
    Settings.llm = HuggingFaceLLM(
        model_name=model, # "Qwen/Qwen3-235B-A22B-Instruct-2507",  # "Qwen/Qwen2.5-7B-Instruct",
        tokenizer_name=tokenizer, # "Qwen/Qwen3-235B-A22B-Instruct-2507",  # "Qwen/Qwen2.5-7B-Instruct",
        context_window=30000,
        max_new_tokens=2000,
        generate_kwargs={
            "temperature": 0.7,
            "top_k": 20,
            "top_p": 0.8,
        },  # {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
        messages_to_prompt=messages_to_prompt,
        completion_to_prompt=completion_to_prompt,
        device_map="auto",
    )

    # Set embedding model
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=embed_model, # "BAAI/bge-multilingual-gemma2"  # "BAAI/bge-base-en-v1.5"
    )

    # Set the size of the text chunk for retrieval
    Settings.transformations = [SentenceSplitter(chunk_size=1024)]

    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

    if input_dir is not None:
        documents = SimpleDirectoryReader(
            input_dir=input_dir
        ).load_data()
    elif input_files is not None:
        documents = SimpleDirectoryReader(
            input_files=input_files
        ).load_data()
    else:
        return
        
    Settings.index = VectorStoreIndex.from_documents(
        documents,
        embed_model=Settings.embed_model,
        transformations=Settings.transformations,
    )

def create_save_vsidx(save_dir, embed_model, input_dir = None, input_files = None):
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=embed_model, # "BAAI/bge-multilingual-gemma2"  # "BAAI/bge-base-en-v1.5"
    )

    Settings.transformations = [SentenceSplitter(chunk_size=1024)]

    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

    if input_dir is not None:
        documents = SimpleDirectoryReader(
            input_dir=input_dir
        ).load_data()
    elif input_files is not None:
        documents = SimpleDirectoryReader(
            input_files=input_files
        ).load_data()
        
    Settings.index = VectorStoreIndex.from_documents(
        documents,
        embed_model=Settings.embed_model,
        transformations=Settings.transformations,
    )
    Settings.index.storage_context.persist(persist_dir=save_dir)
    
def load_vsidx(save_dir):
    from llama_index import StorageContext, load_index_from_storage
    storage_context = StorageContext.from_defaults(persist_dir=save_dir)

    # optional service context
    Settings.index = load_index_from_storage(storage_context)

def query(query_text):
    query_engine = Settings.index.as_query_engine()
    response = query_engine.query(query_text).response
    print(response)
    #return response

In [3]:
create_save_vsidx(
    save_dir="syuu",
    embed_model="BAAI/bge-m3", #"BAAI/bge-multilingual-gemma2",
    input_dir="../translateDirFiles/資治通鑑" )



2026-03-27 20:40:00,878 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


In [ ]:
setup(
    model = "Qwen/Qwen3-14B-FP8", # "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8",
    tokenizer = "Qwen/Qwen3-14B-FP8", #"Qwen/Qwen3-30B-A3B-Instruct-2507-FP8",
    embed_model = "BAAI/bge-m3")

2026-03-27 21:03:43,004 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

2026-03-27 21:04:41,159 - WARNING - Some parameters are on the meta device because they were offloaded to the cpu.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

2026-03-27 21:04:45,757 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


In [9]:
query("匈奴と漢の関係について教えてください。")
    

KeyboardInterrupt: 